# Rooftop Microclimate and PV Power Data Processing Pipeline

**Unified Sequential Architecture**: Data Processing → Quality Assurance → Statistical Analysis

## Processing Stages:

1. **Data Processing (Mandatory)**: Merge → Quality Control → Downsample → Interpolate → Extract
2. **Quality Assurance (Optional)**: Missing rate analysis with visualizations (Stage 6)
3. **Statistical Analysis (Optional)**: Summary statistics, normality testing, and hypothesis tests (Stage 7)

## Configuration:
- **downsampling_config.json**: Frequency and aggregation methods
- **interpolation_config.json**: Gap-filling settings
- **extraction_config.json**: Column selection and output formatting

## PIPELINE EXECUTION OPTIONS & CONFIGURATION

### Toggle optional analysis stages

In [ ]:
# DATA PROCESSING: Set to False to skip optional processing stages
RUN_DOWNSAMPLE = True               # Set to False to skip downsampling (if enabled in config)
RUN_INTERPOLATION = True             # Set to False to skip interpolation (if enabled in config)

# OPTIONAL STAGES: Set to False to skip
RUN_MISSING_RATE_ANALYSIS = True    # Stage 6: Missing rate analysis with heatmaps

### Configure data processing pipeline

In [ ]:
import json
from pathlib import Path
from datetime import datetime
import os

# Optional selection of processed data root for downstream analyses (Stages 6)
# Set to a specific processed folder name (e.g., "20251227_030643_downsampled_true_interpolatetrue")
# or leave as None to auto-select the most recently modified processed_data subfolder.
ANALYSIS_PROCESSED_SUBDIR = None

# Load configuration settings for optional stages
config_dir = Path('../config')
with open(config_dir / 'interpolation_config.json', 'r') as f:
    interp_config = json.load(f)

with open(config_dir / 'downsampling_config.json', 'r') as f:
    downsample_config = json.load(f)

interpolation_enabled = interp_config.get('enabled', False)
downsampling_enabled = downsample_config.get('enabled', True)

# Determine analysis root for Stages 6
analysis_base = Path('../processed_data')
if ANALYSIS_PROCESSED_SUBDIR:
    ANALYSIS_PROCESSED_ROOT = (analysis_base / ANALYSIS_PROCESSED_SUBDIR).resolve()
else:
    # Filter to only timestamped folders (format: YYYYMMDD_HHMMSS_...)
    candidate_dirs = (
        sorted(
            [p for p in analysis_base.iterdir() if p.is_dir() and p.name[0].isdigit() and '_' in p.name],
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if analysis_base.exists()
        else []
)
    ANALYSIS_PROCESSED_ROOT = (candidate_dirs[0] if candidate_dirs else analysis_base).resolve()
os.environ['ANALYSIS_PROCESSED_ROOT'] = str(ANALYSIS_PROCESSED_ROOT)

# Validate configuration: interpolation requires downsampling
if interpolation_enabled and RUN_INTERPOLATION and (not downsampling_enabled or not RUN_DOWNSAMPLE):
    raise ValueError(
        "Interpolation requires downsampling. Cannot run interpolation without downsampling.\n"
        "Please either:\n"
        "  1. Enable and run downsampling (set RUN_DOWNSAMPLE = True)\n"
        "  2. Disable interpolation (set RUN_INTERPOLATION = False)"
    )

# Generate timestamped output directory
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
downsample_flag = 'true' if (downsampling_enabled and RUN_DOWNSAMPLE) else 'false'
interpolate_flag = 'true' if (interpolation_enabled and RUN_INTERPOLATION) else 'false'

OUTPUT_ROOT = Path(f'../processed_data/{timestamp}_downsample_{downsample_flag}_interpolate_{interpolate_flag}')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Prefer the freshly generated output for optional analyses when no override is provided
if ANALYSIS_PROCESSED_SUBDIR is None:
    ANALYSIS_PROCESSED_ROOT = OUTPUT_ROOT.resolve()
    os.environ['ANALYSIS_PROCESSED_ROOT'] = str(ANALYSIS_PROCESSED_ROOT)

# Store in environment for data loading
os.environ['PIPELINE_OUTPUT_ROOT'] = str(OUTPUT_ROOT.resolve())

### Display pipeline configuration

In [ ]:
print("\n" + "="*70)
print("PVIGR DATA ANALYSIS PIPELINE")
print("="*70)
print(f"Output directory: {OUTPUT_ROOT}")
print(f"Configuration: downsample={downsample_flag}, interpolate={interpolate_flag}")
print("\nMandatory stages: Merge → QC → Extract")

# Build stage list
stages = ['Merge', 'QC']
if downsampling_enabled and RUN_DOWNSAMPLE:
    stages.append('Downsample')
if interpolation_enabled and RUN_INTERPOLATION:
    stages.append('Interpolation')
stages.append('Extract')

print(f"Data processing: {' → '.join(stages)}")

# Display optional stages
print("\nOptional analyses:")
print(f"  - Missing Rate Analysis (Stage 6): {'ENABLED' if RUN_MISSING_RATE_ANALYSIS else 'DISABLED'}")

# Display optional stage status
optional_status = []
if downsampling_enabled:
    optional_status.append(f"Downsample: {'Running' if RUN_DOWNSAMPLE else 'SKIPPED'}")
else:
    optional_status.append("Downsample: Disabled in config")
    
if interpolation_enabled:
    optional_status.append(f"Interpolation: {'Running' if RUN_INTERPOLATION else 'SKIPPED'}")
else:
    optional_status.append("Interpolation: Disabled in config")

print(f"\nOptional data processing: {' | '.join(optional_status)}")
print("="*70 + "\n")

## SECTION 1: DATA PROCESSING PIPELINE (MANDATORY)

### Stage 1: Merge Raw Data

In [ ]:
%run 01_merge_raw_data.ipynb
print("✓ Stage 1: Data merging complete")

### Stage 2: Quality Control

In [ ]:
%run 02_quality_control.ipynb
print("✓ Stage 2: Quality control complete")

### Stage 3: Optional Downsampling

In [ ]:
if RUN_DOWNSAMPLE and downsampling_enabled:
    print("Stage 3: Downsampling data...")
    %run 03_downsampling.ipynb
    print("✓ Stage 3: Downsampling complete")
else:
    skip_reason = "Disabled in config" if not downsampling_enabled else "SKIPPED by user"
    print(f"⊘ Stage 3: Downsample - {skip_reason}")

### Stage 4: Optional Interpolation

In [ ]:
if RUN_INTERPOLATION and interpolation_enabled:
    print("Stage 4: Interpolating gaps...")
    %run 04_interpolation.ipynb
    print("✓ Stage 4: Interpolation complete")
else:
    skip_reason = "Disabled in config" if not interpolation_enabled else "SKIPPED by user"
    print(f"⊘ Stage 4: Interpolation - {skip_reason}")

### Stage 5: Data Extraction

In [ ]:
print("Stage 5: Extracting datasets...")
%run 05_extract_data.ipynb
print("✓ Stage 5: Data extraction complete")

print("\n" + "="*70)
print("DATA PROCESSING COMPLETE")
print("="*70)
print("✓ Raw data merged by source type") 
print("✓ Quality control applied with sensor corrections")

if downsampling_enabled and RUN_DOWNSAMPLE:
    print("✓ Data downsampled to uniform temporal resolution")
else:
    print("⊘ Downsampling skipped")

if interpolation_enabled and RUN_INTERPOLATION:
    print("✓ Small gaps filled using interpolation")
else:
    print("⊘ Interpolation skipped")
    
print("✓ Analysis-ready datasets extracted")

print(f"\nOutput Directory: {OUTPUT_ROOT}")
print("\nOutput Structure:")
print(f"  {OUTPUT_ROOT}/merged/              - Raw data combined by source")
print(f"  {OUTPUT_ROOT}/quality_controlled/  - QC'd datasets")
if downsampling_enabled and RUN_DOWNSAMPLE:
    print(f"  {OUTPUT_ROOT}/downsampled/        - Uniform temporal resolution")
if interpolation_enabled and RUN_INTERPOLATION:
    print(f"  {OUTPUT_ROOT}/interpolated/       - Gap-filled datasets")
print(f"  {OUTPUT_ROOT}/extracted/          - Final analysis-ready data")
print("="*70)

## SECTION 2: OPTIONAL QUALITY ASSURANCE - Missing Rate Analysis (Stage 6)

In [ ]:
if RUN_MISSING_RATE_ANALYSIS:
    print("\n" + "="*70)
    print("STAGE 6: MISSING RATE ANALYSIS (OPTIONAL)")
    print("="*70)
    print(f"Looking for data in: {ANALYSIS_PROCESSED_ROOT}")
    
    # Check if extracted data exists
    extracted_path = ANALYSIS_PROCESSED_ROOT / 'extracted'
    if not extracted_path.exists():
        print(f"WARNING: Extracted data folder not found at {extracted_path}")
        print("Stage 6 requires extracted data from Stage 5. Skipping...")
    else:
        print(f"Found extracted data at: {extracted_path}")
        %run 06_missing_rate_analysis.ipynb
        print("✓ Stage 6: Missing rate analysis complete")
else:
    print("\n⊘ Stage 6 (Missing Rate Analysis) - DISABLED")
    print("   To enable, set RUN_MISSING_RATE_ANALYSIS = True")

## Pipeline Complete

In [ ]:
print("\n" + "="*70)
print("ANALYSIS PIPELINE COMPLETE")
print("="*70)
print("\nProcessed Data:")
print(f"  Location: {OUTPUT_ROOT}")

if RUN_MISSING_RATE_ANALYSIS:
    stage6_path = Path(os.environ.get('ANALYSIS_PROCESSED_ROOT', OUTPUT_ROOT)) / 'results' / 'missing_rate_analysis'
    print("\nMissing Rate Analysis (Stage 6):")
    print(f"  Location: {stage6_path}")

print("="*70 + "\n")